NOME: Hercules André
DISCIPLINA: Otimização
Machine Scheduling - Instâncias:

/machinescheduling_instances: Machine Scheduling Instances
Desenvolver em linguagem Julia, usando solver HiGHS. Configuração do solver livre. Isso inclui tempo de execução, gap de otimalidade, tolerâncias.

In [32]:
using JuMP
using HiGHS
using DataFrames

# --- FUNÇÃO GENÉRICA DE SIMULAÇÃO DE HEURÍSTICA ---
function rodar_heuristica(r, p, d, ordem_indices)
    tempo_atual = 0
    atraso_total = 0.0
    for idx in ordem_indices
        if tempo_atual < r[idx]
            tempo_atual = r[idx]
        end
        termino = tempo_atual + p[idx]
        atraso_total += max(0.0, termino - d[idx])
        tempo_atual = termino
    end
    return atraso_total
end

# --- FUNÇÃO PRINCIPAL: RESOLVE UMA INSTÂNCIA E RETORNA A LINHA DE DADOS ---
function resolver_instancia(nome_instancia, n_jobs, r, p, d)
    # 1. Executar Heurísticas
    edd_val  = rodar_heuristica(r, p, d, sortperm(d))
    fifo_val = rodar_heuristica(r, p, d, sortperm(r))
    spt_val  = rodar_heuristica(r, p, d, sortperm(p))
    
    # 2. Configurar e Resolver o MILP
    BigM = sum(r) + sum(p) + 1000
    model = Model(HiGHS.Optimizer)
    set_attribute(model, "time_limit", 30.0)
    set_attribute(model, "output_flag", false)
    
    @variable(model, start[1:n_jobs] >= 0)
    @variable(model, comp[1:n_jobs] >= 0)
    @variable(model, T[1:n_jobs] >= 0)
    @variable(model, z[i in 1:n_jobs, j in 1:n_jobs; i < j], Bin)
    
    for i in 1:n_jobs
        @constraint(model, start[i] >= r[i])
        @constraint(model, comp[i] == start[i] + p[i])
        @constraint(model, T[i] >= comp[i] - d[i])
    end
    
    for i in 1:n_jobs, j in 1:n_jobs
        if i < j
            @constraint(model, start[j] >= comp[i] - BigM * (1 - z[i, j]))
            @constraint(model, start[i] >= comp[j] - BigM * z[i, j])
        end
    end
    
    @objective(model, Min, sum(T[i] for i in 1:n_jobs))
    tempo_exec = @elapsed optimize!(model)
    
    status_solver = string(termination_status(model))
    milp_val = has_values(model) ? objective_value(model) : NaN
    
    # --- CÁLCULO MANUAL CORRIGIDO DO GAP ---
    if has_values(model)
        # Extrai o Lower Bound (Limite Inferior) que o HiGHS encontrou na árvore
        melhor_bound = objective_bound(model)
        
        if milp_val == 0.0
            gap_val = melhor_bound == 0.0 ? 0.0 : Inf
        else
            # Fórmula clássica do GAP percentual relativo: |(Solução - Bound) / Solução|
            gap_val = abs(milp_val - melhor_bound) / abs(milp_val)
        end
    else
        gap_val = NaN
    end
    
    # 3. Lógica para definir a Melhor Regra / Método
    # Criamos um mapeamento para descobrir qual heurística se saiu melhor
    valores = [edd_val, fifo_val, spt_val, milp_val]
    nomes   = ["EDD", "FIFO", "SPT", "MILO"]
    
    # Encontra o menor valor obtido
    menor_valor = minimum(valores)
    
    # Identifica quem atingiu esse menor valor (priorizando as heurísticas se houver empate com o MILO)
    melhor_regra = ""
    if edd_val == menor_valor
        melhor_regra = "EDD"
    elseif fifo_val == menor_valor
        melhor_regra = "FIFO"
    elseif spt_val == menor_valor
        melhor_regra = "SPT"
    else
        melhor_regra = "MILO"
    end
    
    # Se houver múltiplos empates entre heurísticas, indica complacência
    if (edd_val == fifo_val == menor_valor) && melhor_regra != "MILO"
        melhor_regra = "EDD/FIFO"
    elseif (edd_val == spt_val == menor_valor) && melhor_regra != "MILO"
        melhor_regra = "EDD/SPT"
    end

    return (
        instance = nome_instancia,
        n_jobs = n_jobs,
        EDD = edd_val,
        FIFO = fifo_val,
        SPT = spt_val,
        MILO = milp_val,
        gap = round(gap_val * 100, digits=2),
        time = round(tempo_exec, digits=4),
        status = status_solver,
        Melhor_Regra = melhor_regra  # <-- Nova Coluna adicionada aqui!
    )
end

# =================================================================
# BLOCO DE EXECUÇÃO EM MASSA
# =================================================================

df_consolidado = DataFrame(
    instance=String[], n_jobs=Int[], 
    EDD=Float64[], FIFO=Float64[], SPT=Float64[], MILO=Float64[], 
    gap=Float64[], time=Float64[], status=String[],
    Melhor_Regra=String[] # <-- Incluído na estrutura inicial
)

println("Processando instâncias e identificando os melhores métodos...")

# --- INSTÂNCIA: inst_n05_s01 ---
let
    r = [1, 14, 6, 13, 12]; p = [6, 10, 8, 10, 4]; d = [8, 32, 14, 34, 18]
    push!(df_consolidado, resolver_instancia("inst_n05_s01", 5, r, p, d))
end

# --- INSTÂNCIA: inst_n05_s02 ---
let
    r = [5, 8, 11, 4, 14]; p = [5, 6, 10, 3, 6]; d = [11, 23, 24, 13, 20]
    push!(df_consolidado, resolver_instancia("inst_n05_s02", 5, r, p, d))
end

# --- INSTÂNCIA: inst_n05_s03 ---
let
    r = [0, 0, 4, 4, 6]; p = [2, 2, 1, 7, 1]; d = [3, 4, 9, 11, 10]
    push!(df_consolidado, resolver_instancia("inst_n05_s03", 5, r, p, d))
end

# --- INSTÂNCIA: inst_n07_s01 ---
let
    r = [12, 20, 5, 18, 9, 28, 16]; p = [9, 10, 4, 10, 6, 8, 8]; d = [21, 42, 20, 35, 19, 38, 36]
    push!(df_consolidado, resolver_instancia("inst_n07_s01", 7, r, p, d))
end

# --- INSTÂNCIA: inst_n07_s02 ---
let
    r = [0, 11, 9, 3, 4, 5, 5]; p = [6, 5, 8, 6, 4, 5, 9]; d = [8, 24, 27, 17, 14, 18, 25]
    push!(df_consolidado, resolver_instancia("inst_n07_s02", 7, r, p, d))
end

# --- INSTÂNCIA: inst_n07_s03 ---
let
    r = [14, 4, 13, 10, 8, 0, 3]; p = [5, 3, 7, 9, 10, 1, 6]; d = [19, 13, 29, 28, 25, 13, 11]
    push!(df_consolidado, resolver_instancia("inst_n07_s03", 7, r, p, d))
end

# --- INSTÂNCIA: inst_n09_s01 ---
let
    r = [23, 4, 10, 11, 24, 13, 13, 2, 11]; p = [5, 1, 2, 6, 10, 4, 10, 3,6]; d = [33, 16, 20, 27, 40, 20, 33, 19, 22]
    push!(df_consolidado, resolver_instancia("inst_n09_s01", 9, r, p, d))
end

# --- INSTÂNCIA: inst_n09_s02 ---
let
    r = [2, 5, 10, 4, 11, 9, 5, 1, 5]         # release (datas de liberação)
    p = [2, 3, 5, 5, 7, 7, 9, 2, 10]          # duration (tempos de processamento)
    d = [15, 22, 16, 22, 27, 22, 26, 7, 18]   # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n09_s02", 9, r, p, d))
end

# --- INSTÂNCIA: inst_n09_s03 ---
let
    r = [7, 6, 6, 17, 8, 15, 12, 11, 3]       # release (datas de liberação)
    p = [6, 2, 2, 4, 2, 3, 4, 2, 9]           # duration (tempos de processamento)
    d = [22, 14, 15, 31, 13, 24, 22, 14, 12]  # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n09_s03", 9, r, p, d))
end

# --- INSTÂNCIA: inst_n11_s01 ---
let
    r = [6, 8, 16, 16, 3, 1, 5, 19, 0, 1, 19]          # release (datas de liberação)
    p = [7, 2, 1, 6, 3, 3, 1, 10, 1, 8, 3]             # duration (tempos de processamento)
    d = [25, 12, 20, 30, 17, 11, 11, 35, 3, 22, 32]    # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n11_s01", 11, r, p, d))
end

# --- INSTÂNCIA: inst_n11_s02 ---
let
    r = [27, 15, 20, 30, 35, 34, 9, 17, 11, 31, 31]    # release (datas de liberação)
    p = [9, 3, 3, 8, 4, 8, 6, 8, 8, 4, 10]             # duration (tempos de processamento)
    d = [44, 39, 24, 38, 55, 63, 19, 26, 35, 44, 61]    # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n11_s02", 11, r, p, d))
end

# --- INSTÂNCIA: inst_n11_s03 ---
let
    r = [4, 14, 22, 14, 4, 17, 14, 16, 8, 26, 15]      # release (datas de liberação)
    p = [10, 8, 6, 3, 9, 4, 5, 7, 5, 10, 3]            # duration (tempos de processamento)
    d = [24, 32, 35, 18, 34, 32, 28, 26, 27, 41, 27]    # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n11_s03", 11, r, p, d))
end

# --- INSTÂNCIA: inst_n13_s01 ---
let
    r = [13, 25, 11, 24, 14, 18, 18, 24, 28, 23, 12, 8, 10]              # release (datas de liberação)
    p = [10, 2, 2, 6, 2, 5, 1, 7, 8, 7, 3, 6, 2]                         # duration (tempos de processamento)
    d = [25, 35, 13, 43, 34, 24, 29, 43, 39, 46, 29, 29, 22]              # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n13_s01", 13, r, p, d))
end

# --- INSTÂNCIA: inst_n13_s02 ---
let
    r = [18, 35, 16, 22, 9, 32, 19, 10, 16, 14, 39, 35, 2]               # release (datas de liberação)
    p = [8, 10, 1, 5, 9, 1, 7, 8, 6, 10, 2, 3, 8]                         # duration (tempos de processamento)
    d = [31, 63, 23, 29, 32, 55, 31, 26, 23, 45, 44, 43, 32]              # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n13_s02", 13, r, p, d))
end

# --- INSTÂNCIA: inst_n13_s03 ---
let
    r = [22, 38, 5, 28, 21, 0, 3, 38, 21, 23, 38, 6, 20]                 # release (datas de liberação)
    p = [5, 9, 3, 10, 1, 7, 3, 5, 5, 10, 10, 3, 8]                        # duration (tempos de processamento)
    d = [41, 51, 10, 46, 23, 7, 28, 64, 37, 42, 61, 11, 28]               # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n13_s03", 13, r, p, d))
end

# --- INSTÂNCIA: inst_n15_s01 ---
let
    r = [21, 10, 15, 23, 2, 23, 2, 11, 23, 6, 21, 18, 22, 10, 5]          # release (datas de liberação)
    p = [9, 9, 8, 4, 3, 3, 10, 4, 5, 4, 7, 6, 3, 1, 8]                   # duration (tempos de processamento)
    d = [35, 22, 29, 39, 13, 38, 20, 24, 42, 19, 44, 41, 41, 16, 24]      # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n15_s01", 15, r, p, d))
end

# --- INSTÂNCIA: inst_n15_s02 ---
let
    r = [18, 5, 22, 43, 27, 43, 6, 17, 41, 38, 12, 14, 18, 41, 43]        # release (datas de liberação)
    p = [8, 10, 5, 2, 5, 2, 9, 10, 2, 3, 3, 2, 4, 10, 5]                  # duration (tempos de processamento)
    d = [39, 21, 38, 54, 51, 62, 23, 39, 56, 52, 22, 24, 31, 67, 56]      # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n15_s02", 15, r, p, d))
end

# --- INSTÂNCIA: inst_n15_s03 ---
let
    r = [4, 2, 0, 0, 11, 9, 7, 10, 2, 17, 7, 20, 20, 1, 10]               # release (datas de liberação)
    p = [2, 10, 9, 2, 9, 1, 9, 3, 6, 4, 9, 2, 1, 6, 8]                    # duration (tempos de processamento)
    d = [13, 22, 17, 12, 36, 16, 28, 24, 19, 37, 26, 38, 30, 16, 30]      # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n15_s03", 15, r, p, d))
end

# --- INSTÂNCIA: inst_n17_s01 ---
let
    r = [18, 16, 31, 23, 15, 20, 3, 20, 18, 18, 6, 30, 25, 2, 1, 4, 13]   # release (datas de liberação)
    p = [6, 4, 4, 10, 5, 2, 3, 10, 10, 7, 7, 7, 9, 9, 8, 4, 1]            # duration (tempos de processamento)
    d = [44, 27, 49, 46, 30, 38, 16, 50, 48, 38, 25, 52, 54, 28, 21, 15, 22] # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n17_s01", 17, r, p, d))
end

# --- INSTÂNCIA: inst_n17_s02 ---
let
    r = [29, 24, 14, 23, 2, 34, 17, 1, 28, 28, 32, 14, 23, 25, 29, 5, 12] # release (datas de liberação)
    p = [8, 5, 1, 10, 5, 4, 9, 4, 8, 7, 8, 5, 5, 8, 7, 3, 10]             # duration (tempos de processamento)
    d = [54, 45, 26, 52, 23, 50, 39, 11, 49, 54, 53, 29, 46, 43, 58, 22, 32] # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n17_s02", 17, r, p, d))
end

# --- INSTÂNCIA: inst_n17_s03 ---
let
    r = [21, 24, 6, 17, 8, 5, 1, 23, 8, 11, 0, 12, 1, 22, 2, 10, 19]      # release (datas de liberação)
    p = [7, 1, 9, 3, 8, 2, 7, 8, 7, 1, 5, 1, 2, 8, 7, 8, 9]               # duration (tempos de processamento)
    d = [45, 34, 30, 33, 26, 17, 18, 48, 28, 25, 13, 26, 16, 44, 23, 31, 41] # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n17_s03", 17, r, p, d))
end

# --- INSTÂNCIA: inst_n19_s01 ---
let
    r = [29, 39, 21, 25, 4, 18, 30, 24, 11, 23, 23, 10, 44, 30, 16, 20, 28, 22, 6] # release (datas de liberação)
    p = [9, 3, 7, 7, 3, 5, 8, 5, 8, 4, 10, 9, 2, 8, 9, 6, 6, 1, 7]                  # duration (tempos de processamento)
    d = [54, 53, 44, 43, 23, 38, 52, 45, 35, 39, 48, 22, 54, 50, 39, 41, 49, 31, 25] # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n19_s01", 19, r, p, d))
end

# --- INSTÂNCIA: inst_n19_s02 ---
let
    r = [44, 43, 6, 41, 14, 14, 18, 38, 27, 43, 2, 41, 18, 43, 35, 17, 22, 5, 12] # release (datas de liberação)
    p = [2, 5, 9, 10, 2, 5, 4, 3, 5, 2, 8, 2, 8, 5, 10, 10, 5, 10, 3]               # duration (tempos de processamento)
    d = [54, 56, 23, 67, 24, 31, 31, 52, 51, 62, 32, 56, 39, 56, 63, 39, 38, 21, 22] # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n19_s02", 19, r, p, d))
end

# --- INSTÂNCIA: inst_n19_s03 ---
let
    r = [2, 10, 11, 20, 1, 15, 0, 7, 10, 4, 17, 7, 20, 2, 0, 26, 9, 8, 14]       # release (datas de liberação)
    p = [6, 3, 9, 2, 6, 3, 9, 9, 8, 2, 4, 9, 1, 10, 2, 10, 1, 5, 5]               # duration (tempos de processamento)
    d = [19, 24, 36, 38, 16, 27, 17, 28, 30, 13, 37, 26, 30, 22, 12, 41, 16, 27, 28] # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n19_s03", 19, r, p, d))
end

# --- INSTÂNCIA: inst_n21_s01 ---
let
    r = [26, 44, 43, 6, 41, 14, 14, 18, 38, 27, 43, 2, 41, 18, 43, 35, 17, 22, 5, 12, 16] # release (datas de liberação)
    p = [5, 2, 5, 9, 10, 2, 5, 4, 3, 5, 2, 8, 2, 8, 5, 10, 10, 5, 10, 3, 5]               # duration (tempos de processamento)
    d = [39, 54, 56, 23, 67, 24, 31, 31, 52, 51, 62, 32, 56, 39, 56, 63, 39, 38, 21, 22, 28] # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n21_s01", 21, r, p, d))
end

# --- INSTÂNCIA: inst_n21_s02 ---
let
    r = [14, 17, 24, 25, 22, 19, 33, 1, 2, 27, 21, 11, 23, 23, 10, 44, 30, 16, 20, 28, 6] # release (datas de liberação)
    p = [9, 8, 10, 3, 4, 9, 7, 7, 3, 5, 8, 5, 8, 4, 10, 9, 2, 8, 9, 6, 7]                  # duration (tempos de processamento)
    d = [27, 39, 53, 54, 35, 39, 52, 13, 23, 38, 52, 45, 35, 39, 48, 22, 54, 50, 39, 41, 25] # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n21_s02", 21, r, p, d))
end

# --- INSTÂNCIA: inst_n21_s03 ---
let
    r = [2, 10, 11, 20, 1, 15, 0, 7, 10, 4, 17, 7, 20, 2, 0, 26, 9, 8, 14, 21, 22]       # release (datas de liberação)
    p = [6, 3, 9, 2, 6, 3, 9, 9, 8, 2, 4, 9, 1, 10, 2, 10, 1, 5, 5, 9, 3]                  # duration (tempos de processamento)
    d = [19, 24, 36, 38, 16, 27, 17, 28, 30, 13, 37, 26, 30, 22, 12, 41, 16, 27, 28, 43, 37] # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n21_s03", 21, r, p, d))
end

# --- INSTÂNCIA: inst_n23_s01 ---
let
    r = [29, 39, 21, 25, 4, 18, 30, 24, 11, 23, 23, 10, 44, 30, 16, 20, 28, 22, 6, 26, 44, 43, 6] # release (datas de liberação)
    p = [9, 3, 7, 7, 3, 5, 8, 5, 8, 4, 10, 9, 2, 8, 9, 6, 6, 1, 7, 5, 2, 5, 9]                  # duration (tempos de processamento)
    d = [54, 53, 44, 43, 23, 38, 52, 45, 35, 39, 48, 22, 54, 50, 39, 41, 49, 31, 25, 39, 54, 56, 23] # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n23_s01", 23, r, p, d))
end

# --- INSTÂNCIA: inst_n23_s02 ---
let
    r = [14, 14, 18, 38, 27, 43, 2, 41, 18, 43, 35, 17, 22, 5, 12, 14, 17, 24, 25, 22, 19, 33, 1] # release (datas de liberação)
    p = [2, 5, 4, 3, 5, 2, 8, 2, 8, 5, 10, 10, 5, 10, 3, 9, 8, 10, 3, 4, 9, 7, 7]                  # duration (tempos de processamento)
    d = [24, 31, 31, 52, 51, 62, 32, 56, 39, 56, 63, 39, 38, 21, 22, 27, 39, 53, 54, 35, 39, 52, 13] # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n23_s02", 23, r, p, d))
end

# --- INSTÂNCIA: inst_n23_s03 ---
let
    r = [2, 27, 21, 11, 23, 23, 10, 44, 30, 16, 20, 28, 6, 2, 10, 11, 20, 1, 15, 0, 7, 10, 4]       # release (datas de liberação)
    p = [3, 5, 8, 5, 8, 4, 10, 9, 2, 8, 9, 6, 7, 6, 3, 9, 2, 6, 3, 9, 9, 8, 2]                  # duration (tempos de processamento)
    d = [23, 38, 52, 45, 35, 39, 48, 22, 54, 50, 39, 41, 25, 19, 24, 36, 38, 16, 27, 17, 28, 30, 13] # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_n23_s03", 23, r, p, d))
end

# --- INSTÂNCIA: inst_book ---
let
    r = [2, 5, 1, 4, 7, 3, 4]       # release (datas de liberação)
    p = [3, 2, 2, 1, 4, 4, 3]       # duration (tempos de processamento)
    d = [6, 9, 8, 5, 12, 10, 9]     # due (prazos limites)
    push!(df_consolidado, resolver_instancia("inst_book", 7, r, p, d))
end

println("Concluído!")
println("\n=========================================================================================")
println("                   TABELA RESUMO DE MÁQUINA ÚNICA COM MELHOR REGRA                       ")
println("=========================================================================================")
println(df_consolidado)
println("=========================================================================================\n")

Processando instâncias e identificando os melhores métodos...
Concluído!

                   TABELA RESUMO DE MÁQUINA ÚNICA COM MELHOR REGRA                       
31×10 DataFrame
 Row │ instance      n_jobs  EDD      FIFO     SPT      MILO     gap      time     status      Melhor_Regra 
     │ String        Int64   Float64  Float64  Float64  Float64  Float64  Float64  String      String       
─────┼──────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ inst_n05_s01       5      7.0      9.0     54.0      7.0     0.0    0.0514  OPTIMAL     MILO
   2 │ inst_n05_s02       5     15.0     19.0     15.0     15.0     0.0    0.1763  OPTIMAL     MILO
   3 │ inst_n05_s03       5      3.0      4.0     20.0      3.0     0.0    0.0868  OPTIMAL     EDD
   4 │ inst_n07_s01       7     58.0     42.0     99.0     38.0     0.0    0.4657  OPTIMAL     MILO
   5 │ inst_n07_s02       7     31.0     40.0     77.0     30.0     0.0    0.7592  OPTIMAL    